In [2]:
import numpy as np


grid = [
    [' ', ' ', ' ', 'G'],
    [' ', 'X', ' ', 'P'],
    [' ', ' ', ' ', ' ']
]

rows, cols = len(grid), len(grid[0])

gamma = 0.9
theta = 1e-4

actions = [(-1,0), (1,0), (0,-1), (0,1)]
directions = ['↑', '↓', '←', '→']


def is_valid(r, c):
    return 0 <= r < rows and 0 <= c < cols and grid[r][c] != 'X'

def reward(r, c):
    if grid[r][c] == 'G':
        return 10
    elif grid[r][c] == 'P':
        return -10
    return -1

def print_values(iteration, V):
    print(f"\nIteration {iteration} - Value Function:")
    print(np.round(V, 2))

def print_policy(policy):
    for row in policy:
        print(row)


def value_iteration():
    V = np.zeros((rows, cols))
    iteration = 0

    while True:
        iteration += 1
        delta = 0
        new_V = np.copy(V)

        for r in range(rows):
            for c in range(cols):

                if grid[r][c] in ['G', 'P', 'X']:
                    continue

                values = []
                for a in actions:
                    nr, nc = r + a[0], c + a[1]

                    if not is_valid(nr, nc):
                        nr, nc = r, c

                    values.append(reward(nr, nc) + gamma * V[nr][nc])

                best = max(values)
                new_V[r][c] = best
                delta = max(delta, abs(V[r][c] - best))

        V = new_V
        print_values(iteration, V)

        if delta < theta:
            print("\nValue Iteration Converged!\n")
            break

    return V

def extract_policy(V):
    policy = [['' for _ in range(cols)] for _ in range(rows)]

    for r in range(rows):
        for c in range(cols):

            if grid[r][c] == 'X':
                policy[r][c] = 'X'
                continue
            if grid[r][c] in ['G', 'P']:
                policy[r][c] = grid[r][c]
                continue

            values = []
            for i, a in enumerate(actions):
                nr, nc = r + a[0], c + a[1]

                if not is_valid(nr, nc):
                    nr, nc = r, c

                values.append(reward(nr, nc) + gamma * V[nr][nc])

            policy[r][c] = directions[np.argmax(values)]

    return policy


def policy_iteration():
    policy = np.random.choice(directions, (rows, cols))
    V = np.zeros((rows, cols))

    iteration = 0

    while True:
        iteration += 1
        print(f"\nPolicy Iteration Step {iteration}")

        # 🔸 Policy Evaluation
        while True:
            delta = 0
            new_V = np.copy(V)

            for r in range(rows):
                for c in range(cols):

                    if grid[r][c] in ['G', 'P', 'X']:
                        continue

                    a = directions.index(policy[r][c])
                    nr, nc = r + actions[a][0], c + actions[a][1]

                    if not is_valid(nr, nc):
                        nr, nc = r, c

                    val = reward(nr, nc) + gamma * V[nr][nc]
                    new_V[r][c] = val
                    delta = max(delta, abs(V[r][c] - val))

            V = new_V

            if delta < theta:
                break

        print("Value Function:")
        print(np.round(V, 2))

        # 🔸 Policy Improvement
        stable = True

        for r in range(rows):
            for c in range(cols):

                if grid[r][c] in ['G', 'P', 'X']:
                    continue

                old_action = policy[r][c]

                values = []
                for i, a in enumerate(actions):
                    nr, nc = r + a[0], c + a[1]

                    if not is_valid(nr, nc):
                        nr, nc = r, c

                    values.append(reward(nr, nc) + gamma * V[nr][nc])

                best_action = directions[np.argmax(values)]
                policy[r][c] = best_action

                if old_action != best_action:
                    stable = False

        print("Policy:")
        print_policy(policy)

        if stable:
            print("\nPolicy Iteration Converged!\n")
            break

    return policy, V


print("\n===== VALUE ITERATION =====")
V = value_iteration()

vi_policy = extract_policy(V)
print("Final Policy from Value Iteration:")
print_policy(vi_policy)

print("\n===== POLICY ITERATION =====")
pi_policy, pi_values = policy_iteration()

print("\nFinal Policy from Policy Iteration:")
print_policy(pi_policy)



===== VALUE ITERATION =====

Iteration 1 - Value Function:
[[-1. -1. 10.  0.]
 [-1.  0. -1.  0.]
 [-1. -1. -1. -1.]]

Iteration 2 - Value Function:
[[-1.9  8.  10.   0. ]
 [-1.9  0.   8.   0. ]
 [-1.9 -1.9 -1.9 -1.9]]

Iteration 3 - Value Function:
[[ 6.2   8.   10.    0.  ]
 [-2.71  0.    8.    0.  ]
 [-2.71 -2.71  6.2  -2.71]]

Iteration 4 - Value Function:
[[ 6.2   8.   10.    0.  ]
 [ 4.58  0.    8.    0.  ]
 [-3.44  4.58  6.2   4.58]]

Iteration 5 - Value Function:
[[ 6.2   8.   10.    0.  ]
 [ 4.58  0.    8.    0.  ]
 [ 3.12  4.58  6.2   4.58]]

Iteration 6 - Value Function:
[[ 6.2   8.   10.    0.  ]
 [ 4.58  0.    8.    0.  ]
 [ 3.12  4.58  6.2   4.58]]

Value Iteration Converged!

Final Policy from Value Iteration:
['→', '→', '→', 'G']
['↑', 'X', '↑', 'P']
['↑', '→', '↑', '←']

===== POLICY ITERATION =====

Policy Iteration Step 1
Value Function:
[[-10. -10. -10.   0.]
 [-10.   0. -10.   0.]
 [-10. -10. -10. -10.]]
Policy:
['↑' '↑' '→' '↓']
['↑' '↓' '↑' '↑']
['↑' '↑' '↓' '↓']